# baseline_rag_colab.ipynb — Baseline RAG Pipeline (Google Colab A100)
**Project:** Self-Corrective RAG (CRAG) — ITMD 524, Spring 2026\
**Purpose:** Run the full baseline RAG evaluation on Colab A100 with a local Llama 3.1-8B generator.\
**Environment:** Google Colab Pro (A100 GPU)

## Before running this notebook
1. Ensure your runtime is connected to an **A100 GPU** (Runtime -> Change runtime type).
2. Upload your `crag-vectors` folder to the root of your Google Drive (`MyDrive/crag-vectors`).


In [ ]:
!pip install -q \
faiss-cpu \
sentence-transformers \
transformers \
accelerate \
datasets \
ragas \
langchain-groq \
langchain-community \
langchain-huggingface \
python-dotenv \
requests==2.32.4

In [ ]:
# 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configurar Rutas (Apuntando a Google Drive)
import os

ARTIFACTS_DIR = "/content/drive/MyDrive/crag-vectors"
INDEX_PATH    = f"{ARTIFACTS_DIR}/corpus_index.faiss"
IDMAP_PATH    = f"{ARTIFACTS_DIR}/id_map.json"
CONFIG_PATH   = f"{ARTIFACTS_DIR}/config.json"
RESULTS_PATH  = "/content/drive/MyDrive/baseline_results.csv"

ENCODER_MODEL   = "all-MiniLM-L6-v2"
GENERATOR_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
TOP_K           = 5
EVAL_SAMPLES    = 800
HF_DATASET_ID   = "AdamLucek/apple-environmental-report-QA-retrieval"

print(f"Artifacts dir   : {ARTIFACTS_DIR}")
print(f"Encoder model   : {ENCODER_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"TOP_K           : {TOP_K}")
print(f"Eval samples    : {EVAL_SAMPLES}")

In [ ]:
# 4. Verificar Entorno
import faiss, sentence_transformers, transformers, torch, datasets, ragas
print(f"faiss                 {faiss.__version__}")
print(f"sentence-transformers {sentence_transformers.__version__}")
print(f"transformers          {transformers.__version__}")
print(f"torch                 {torch.__version__}")
print(f"GPU available         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                   {torch.cuda.get_device_name(0)}")

In [ ]:
# 5. Cargar API Key desde los secretos de Colab
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

assert GROQ_API_KEY, "GROQ_API_KEY not found in Colab Secrets."
print(f"Groq API key loaded (starts with: {GROQ_API_KEY[:8]}...)")

In [ ]:
# 6. Cargar FAISS Index e ID Map
import faiss
import json

index  = faiss.read_index(INDEX_PATH)
id_map = json.load(open(IDMAP_PATH, encoding="utf-8"))

assert index.ntotal == len(id_map), (
    f"Mismatch: index has {index.ntotal} vectors but id_map has {len(id_map)} entries."
)

print(f"FAISS index loaded: {index.ntotal} vectors, dim={index.d}")
print(f"id_map loaded     : {len(id_map)} entries")

In [ ]:
# 7. Encoder y Retrieve
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer(ENCODER_MODEL)

def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    q_vec = encoder.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        entry = id_map[str(idx)]
        results.append({
            "text":     entry["text"],
            "chunk_id": entry["chunk_id"],
            "score":    float(dist),
        })
    return results

sample = retrieve("Apple renewable energy percentage", top_k=3)
print(f"retrieve() — {len(sample)} results")

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading {GENERATOR_MODEL} on GPU...")
gen_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL,
    token=HF_TOKEN
)

gen_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

gen_model.eval()

SYSTEM_PROMPT = "You are a helpful assistant. Answer the question based on the provided context."

def generate(query: str, context_chunks: list[dict]) -> str:
    context  = "\n\n".join(c["text"] for c in context_chunks)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    inputs = gen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(gen_model.device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=gen_tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

answer = generate("What is Apple's renewable energy percentage?", sample)
print("generate() — ok")

In [ ]:
# 9. Full Pipeline baseline_rag()
def baseline_rag(query: str, top_k: int = TOP_K) -> dict:
    chunks = retrieve(query, top_k=top_k)
    answer = generate(query, chunks)
    return {
        "answer":   answer,
        "contexts": [c["text"] for c in chunks],
        "chunks":   chunks,
    }

queries = [
    "How does Apple track its carbon emissions?",
]

for q in queries:
    result = baseline_rag(q)
    print(f"Q: {q}\nA: {result['answer'][:300]}\n")

In [ ]:
# 10. Batched Evaluation (Optimizada para A100, BATCH_SIZE=16)
import pandas as pd
import time
import torch
from datasets import load_dataset

ds  = load_dataset(HF_DATASET_ID)
val = pd.DataFrame(ds["validation"]).drop_duplicates(subset="question")
val = val.sample(n=EVAL_SAMPLES, random_state=42).reset_index(drop=True)

print(f"Running batched evaluation on {EVAL_SAMPLES} samples...")

records = []
start   = time.time()
BATCH_SIZE = 16  # Se sube a 16 porque la A100 tiene 40GB de VRAM

if getattr(gen_tokenizer, 'pad_token', None) is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_tokenizer.padding_side = 'left'

for i in range(0, len(val), BATCH_SIZE):
    batch = val.iloc[i:i+BATCH_SIZE]

    batch_contexts = []
    batch_chunks = []
    for q in batch["question"]:
        chunks = retrieve(q, top_k=TOP_K)
        batch_chunks.append(chunks)
        batch_contexts.append([c["text"] for c in chunks])

    batch_messages = []
    for q, chunks in zip(batch["question"], batch_chunks):
        context = "\n\n".join([c["text"] for c in chunks])
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {q}"},
        ]
        batch_messages.append(messages)

    texts = [gen_tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in batch_messages]
    inputs = gen_tokenizer(texts, padding=True, return_tensors="pt", add_special_tokens=False).to(gen_model.device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=gen_tokenizer.pad_token_id,
        )

    new_tokens = output_ids[:, inputs["input_ids"].shape[-1]:]
    answers = gen_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for j, (idx, row) in enumerate(batch.iterrows()):
        records.append({
            "question":     row["question"],
            "ground_truth": row["chunk"],
            "answer":       answers[j].strip(),
            "contexts":     batch_contexts[j],
        })

    processed = i + len(batch)
    elapsed = time.time() - start
    avg = elapsed / processed
    remaining = avg * (EVAL_SAMPLES - processed)
    print(f"[{processed:03d}/{EVAL_SAMPLES}] Batched elapsed: {elapsed/60:.1f} min | remaining: {remaining/60:.1f} min")

    if processed % 160 == 0 or processed == len(val):
        pd.DataFrame(records).to_csv(RESULTS_PATH, index=False)

pd.DataFrame(records).to_csv(RESULTS_PATH, index=False)
print(f"\nBatch complete. {len(records)} samples saved to {RESULTS_PATH}\n")

In [ ]:
# 11. Install vLLM + RAGAS dependencies
!pip install -q vllm ragas langchain langchain-openai langchain-huggingface

In [ ]:
# 12. Free GPU memory from generator (we already have baseline_results.csv)
import gc, torch

try:
    del gen_model, gen_tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# 13. RAGAS Evaluation -- vLLM + Qwen2.5-7B-Instruct (A100 optimized)
# ==============================================================================
# Fix: Replaces broken Zephyr-7B HuggingFace pipeline with vLLM server.
# - vLLM provides continuous batching (10-50x faster)
# - Qwen2.5-7B-Instruct has much better JSON compliance
# - Expected: ~30-50 min for 800 samples (vs 20+ hours before)
# ==============================================================================
import subprocess, time, os, ast, requests
import pandas as pd
from datasets import Dataset

JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
VLLM_PORT   = 8000

# ---------- 1. Start vLLM server ----------
# Log to file so we can see errors if startup fails
VLLM_LOG = '/content/vllm_server.log'
print(f"Starting vLLM server with {JUDGE_MODEL}...")
vllm_log_fh = open(VLLM_LOG, 'w')
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", JUDGE_MODEL,
        "--dtype", "bfloat16",
        "--gpu-memory-utilization", "0.45",
        "--max-model-len", "8192",
        "--port", str(VLLM_PORT),
        "--no-enable-log-requests",
    ],
    stdout=vllm_log_fh,
    stderr=vllm_log_fh,
)

# Wait up to 300s (first run downloads ~15GB of weights)
for i in range(300):
    # Check if process died
    if vllm_proc.poll() is not None:
        vllm_log_fh.close()
        print(open(VLLM_LOG).read()[-2000:])
        raise RuntimeError(f"vLLM exited with code {vllm_proc.returncode}. See log above.")
    try:
        r = requests.get(f"http://localhost:{VLLM_PORT}/health")
        if r.status_code == 200:
            print(f"  vLLM ready after {i+1}s")
            break
    except Exception:
        pass
    if i % 30 == 29:
        print(f"  Still waiting... {i+1}s")
    time.sleep(1)
else:
    vllm_log_fh.close()
    print('--- vLLM log (last 2000 chars) ---')
    print(open(VLLM_LOG).read()[-2000:])
    raise RuntimeError("vLLM server did not start within 300s. See log above.")

# ---------- 2. Load baseline results ----------
results_df = pd.read_csv(RESULTS_PATH)

def safe_eval(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except Exception:
        return []

results_df["contexts"] = results_df["contexts"].apply(safe_eval)
results_df = results_df.dropna(subset=["question", "answer", "ground_truth"])
results_df = results_df[results_df["contexts"].map(len) > 0].reset_index(drop=True)

ragas_data = Dataset.from_list([
    {
        "question":     row["question"],
        "answer":       row["answer"],
        "contexts":     row["contexts"],
        "ground_truth": row["ground_truth"],
    }
    for _, row in results_df.iterrows()
])
print(f"Evaluating {len(ragas_data)} samples.")

# ---------- 3. Configure RAGAS judge ----------
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy
from ragas import evaluate
from ragas.run_config import RunConfig

judge_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=JUDGE_MODEL,
        base_url=f"http://localhost:{VLLM_PORT}/v1",
        api_key="not-needed",
        temperature=0,
        max_tokens=2048,
    )
)

judge_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name=f"sentence-transformers/{ENCODER_MODEL}",
        model_kwargs={"device": "cuda"},
        encode_kwargs={"batch_size": 64},
    )
)

# ---------- 4. Run RAGAS ----------
run_cfg = RunConfig(
    timeout=300,
    max_retries=3,
    max_wait=30,
    max_workers=4,
)

print("Running RAGAS evaluation...")
t0 = time.time()

scores = evaluate(
    dataset=ragas_data,
    metrics=[faithfulness, answer_relevancy],
    llm=judge_llm,
    embeddings=judge_emb,
    run_config=run_cfg,
    raise_exceptions=False,
)

elapsed = time.time() - t0
print(f"RAGAS completed in {elapsed/60:.1f} min")

# ---------- 5. Results ----------
scores_df = scores.to_pandas()
n_valid_faith = scores_df["faithfulness"].notna().sum()
n_valid_relev = scores_df["answer_relevancy"].notna().sum()

faithfulness_score     = scores_df["faithfulness"].dropna().mean()
answer_relevancy_score = scores_df["answer_relevancy"].dropna().mean()

print(f"\n{'='*55}")
print(f"  Baseline RAG -- RAGAS Results  (n={len(ragas_data)} samples)")
print(f"{'='*55}")
print(f"  Faithfulness       : {faithfulness_score:.4f}  (valid={n_valid_faith})")
print(f"  Answer Relevancy   : {answer_relevancy_score:.4f}  (valid={n_valid_relev})")
print(f"  Hallucination rate : {1 - faithfulness_score:.4f}")
print(f"{'='*55}")

RAGAS_OUT = RESULTS_PATH.replace(".csv", "_ragas.csv")
scores_df.to_csv(RAGAS_OUT, index=False)
print(f"\nPer-sample scores saved -> {RAGAS_OUT}")

# ---------- 6. Cleanup ----------
vllm_proc.terminate()
print("vLLM server stopped.")